In [2]:
import os
import cv2
import numpy as np


def create_dataset(video_dir, label, frame_count=10, frame_size=(256, 256)):
    videos = []
    labels = []
    i = 0
    for video_file in os.listdir(video_dir):
        video_path = os.path.join(video_dir, video_file)
        cap = cv2.VideoCapture(video_path)
        frames = []
        i += 1
        print(i, end=" ")
        if i == 1000:
            break
        # Read the specified number of frames
        while len(frames) < frame_count:
            ret, frame = cap.read()
            if not ret:
                break
            # Resize frame
            frame = cv2.resize(frame, frame_size)
            frames.append(frame)

        cap.release()

        # Only add the video if it has the required number of frames
        if len(frames) == frame_count:
            videos.append(np.array(frames))  # Convert frames to NumPy array
            labels.append(label)

    return videos, labels  # Return as lists, not as NumPy arrays


# Example usage:
real_videos_dir = 'celeb-df-v2/Celeb-real'
fake_videos_dir = 'celeb-df-v2/Celeb-synthesis'

# Create datasets
real_videos, real_labels = create_dataset(real_videos_dir, label=1)
fake_videos, fake_labels = create_dataset(fake_videos_dir, label=0)

# Combine real and fake datasets
videos = real_videos + fake_videos
labels = real_labels + fake_labels

# Convert lists to NumPy arrays after combining
videos = np.array(videos)
labels = np.array(labels)

print(f"Shape of video dataset: {videos.shape}")
print(f"Shape of labels: {labels.shape}")

1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 266 267 268 269 270 271 272 273 274 275 276 277 

In [ ]:
# import tensorflow as tf

# # Ensure TensorFlow uses the GPU
# physical_devices = tf.config.list_physical_devices('GPU')
# if physical_devices:
#     try:
#         tf.config.experimental.set_memory_growth(physical_devices[0], True)
#         print(f"Using GPU: {physical_devices[0]}")
#     except RuntimeError as e:
#         print(e)

# train_data = tf.data.Dataset.from_tensor_slices((videos, labels)).batch(1)

print(train_data)

<BatchDataset element_spec=(TensorSpec(shape=(None, 10, 256, 256, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv3D, BatchNormalization, MaxPooling3D, Flatten, Dense, Dropout
from tensorflow.keras.models import Model

# Ensure TensorFlow uses the GPU
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
        print(f"Using GPU: {physical_devices[0]}")
    except RuntimeError as e:
        print(e)

class DeepFakeDetectionModel:
    def __init__(self, input_shape):
        self.input_shape = input_shape
        self.model = self.build_model()

    def build_model(self):
        inputs = Input(shape=self.input_shape)

        # Convolutional Layer 1
        x = Conv3D(8, (3, 3, 3), activation='relu', padding='same', name='conv3d_1')(inputs)
        x = BatchNormalization(name='batch_normalization_1')(x)
        x = MaxPooling3D((2, 2, 2), name='max_pooling3d_1')(x)

        # Convolutional Layer 2
        x = Conv3D(16, (3, 3, 3), activation='relu', padding='same', name='conv3d_2')(x)
        x = BatchNormalization(name='batch_normalization_2')(x)
        x = MaxPooling3D((2, 2, 2), name='max_pooling3d_2')(x)

        # Convolutional Layer 3
        x = Conv3D(32, (3, 3, 3), activation='relu', padding='same', name='conv3d_3')(x)
        x = BatchNormalization(name='batch_normalization_3')(x)
        x = MaxPooling3D((2, 2, 2), name='max_pooling3d_3')(x)

        # Flatten
        x = Flatten(name='flatten')(x)

        # Dense Layer
        x = Dense(64, activation='relu', name='dense_1')(x)
        x = Dropout(0.5, name='dropout')(x)

        # Output Layer
        outputs = Dense(1, activation='sigmoid', name='output')(x)

        # Create Model
        model = Model(inputs=inputs, outputs=outputs, name='DeepFakeDetectionModel')

        # Compile Model
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        return model

    def summary(self):
        return self.model.summary()

    def train(self, train_data, validation_data, epochs=150, batch_size=16):
        history = self.model.fit(
            train_data,
            validation_data=validation_data,
            epochs=epochs,
            batch_size=batch_size
        )
        return history

    def evaluate(self, test_data):
        return self.model.evaluate(test_data)

    def predict(self, data):
        return self.model.predict(data)

    def save(self, filepath):
        self.model.save(filepath)

    def load(self, filepath):
        self.model = tf.keras.models.load_model(filepath)

# Example usage:
# Define input shape according to your data
frame_count = 10
height = width = 256
channels = 3
input_shape = (frame_count, height, width, channels)  # (10, 256, 256, 3)

# Initialize the model
deepfake_detector = DeepFakeDetectionModel(input_shape)

# Print model summary
deepfake_detector.summary()

# Prepare your data
# Create tf.data.Dataset or use numpy arrays directly
train_data = tf.data.Dataset.from_tensor_slices((videos, labels)).batch(16)
validation_data = tf.data.Dataset.from_tensor_slices((videos, labels)).batch(16)  # Replace with actual validation data

# Train the model
deepfake_detector.train(train_data, validation_data, epochs=28)

# Evaluate the model (example)
# deepfake_detector.evaluate(test_data)

# Save the model (example)
deepfake_detector.save('deepfake_detection_model.h5')

# Load the model (example)
deepfake_detector.load('deepfake_detection_model.h5')


Model: "DeepFakeDetectionModel"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 10, 256, 256, 3)  0         
                             ]                                   
                                                                 
 conv3d_1 (Conv3D)           (None, 10, 256, 256, 8)   656       
                                                                 
 batch_normalization_1 (Batc  (None, 10, 256, 256, 8)  32        
 hNormalization)                                                 
                                                                 
 max_pooling3d_1 (MaxPooling  (None, 5, 128, 128, 8)   0         
 3D)                                                             
                                                                 
 conv3d_2 (Conv3D)           (None, 5, 128, 128, 16)   3472      
                                            